# Kepler Vetting Final Report

Quick report notebook. It reads the saved CSVs from `outputs/metrics/split801010/` and rebuilds the final tables/plots.

No training here.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, Markdown


def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "src" / "kepler_vetting").exists():
            return candidate
    return cwd


ROOT = find_repo_root()
METRICS_DIR = ROOT / "outputs" / "metrics" / "split801010"

MODEL_COMPARISON_PATH = METRICS_DIR / "model_comparison.csv"
MODEL_COMPARISON_BY_SEED_PATH = METRICS_DIR / "model_comparison_by_seed.csv"
PAIRWISE_SUMMARY_PATH = METRICS_DIR / "pairwise_model_error_summary.csv"
FUSED_PREDICTIONS_PATH = METRICS_DIR / "fused_local_model_predictions.csv"

required_paths = [
    MODEL_COMPARISON_PATH,
    MODEL_COMPARISON_BY_SEED_PATH,
    PAIRWISE_SUMMARY_PATH,
    FUSED_PREDICTIONS_PATH,
]

missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        "Missing required report inputs:\n"
        + "\n".join(str(path) for path in missing_paths)
    )

comparison = pd.read_csv(MODEL_COMPARISON_PATH)
comparison_by_seed = pd.read_csv(MODEL_COMPARISON_BY_SEED_PATH)
pairwise = pd.read_csv(PAIRWISE_SUMMARY_PATH)
fused_predictions = pd.read_csv(FUSED_PREDICTIONS_PATH)

display(Markdown(f"**Repo root:** `{ROOT}`"))
display(Markdown(f"**Metrics directory:** `{METRICS_DIR}`"))

## Dataset and split summary

In [ ]:
unique_rows = fused_predictions.drop_duplicates("row_index").copy()

dataset_summary = pd.DataFrame(
    [
        {"item": "Total rows", "count": unique_rows.shape[0]},
        {
            "item": "False positive",
            "count": int((unique_rows["disposition"] == "FALSE POSITIVE").sum()),
        },
        {
            "item": "Candidate",
            "count": int((unique_rows["disposition"] == "CANDIDATE").sum()),
        },
        {
            "item": "Confirmed",
            "count": int((unique_rows["disposition"] == "CONFIRMED").sum()),
        },
        {
            "item": "Planet-like total, candidate + confirmed",
            "count": int((unique_rows["y_true"] == 1).sum()),
        },
    ]
)

display(dataset_summary)

seed0 = fused_predictions[fused_predictions["seed"] == 0].copy()
split_summary = (
    seed0.groupby("split")
    .agg(
        rows=("row_index", "count"),
        groups=("kepid", "nunique"),
        negative=("y_true", lambda values: int((values == 0).sum())),
        positive=("y_true", lambda values: int((values == 1).sum())),
    )
    .reset_index()
)
split_summary["positive_rate"] = split_summary["positive"] / split_summary["rows"]
split_order = {"train": 0, "val": 1, "test": 2}
split_summary = split_summary.sort_values(
    "split", key=lambda series: series.map(split_order)
).reset_index(drop=True)

display(split_summary)

## Final model verdict

Final model: `fused_tabular_local_cnn`, using the 80/10/10 grouped split and fixed threshold 0.5.

In [ ]:
FIXED_VARIANT = "fixed_threshold_0.5"
TUNED_VARIANT = "val_tuned_f1_threshold"
FINAL_MODEL = "fused_tabular_local_cnn"

metric_columns = [
    "threshold_mean",
    "accuracy_mean",
    "f1_mean",
    "roc_auc_mean",
    "precision_mean",
    "recall_mean",
]

final_row = comparison[
    (comparison["display_model"] == FINAL_MODEL)
    & (comparison["metric_variant"] == FIXED_VARIANT)
    & (comparison["split"] == "test")
][["display_model", "metric_variant", "split", *metric_columns]].copy()

display(final_row.round(3))

## Main fixed-threshold comparison

In [ ]:
model_order = [
    "dummy_most_frequent",
    "tabular_logistic_regression",
    "tabular_local_features_logistic_regression",
    "local_view_cnn",
    "global_view_cnn",
    "fused_tabular_local_cnn",
    "soft_label_fused_tabular_local_cnn",
    "candidate_weighted_fused_tabular_local_cnn",
    "three_class_fused_tabular_local_cnn",
    "two_stage_rescue_gate",
    "fused_tabular_transit_set_cnn",
    "fused_tabular_local_transit_set_cnn",
    "rescue_stacked_logistic_regression",
    "selective_rescue_rule_model",
]

fixed_comparison = comparison[
    (comparison["metric_variant"] == FIXED_VARIANT)
    & (comparison["split"] == "test")
    & (comparison["display_model"].isin(model_order))
].copy()

fixed_comparison["model_order"] = fixed_comparison["display_model"].map(
    {model: idx for idx, model in enumerate(model_order)}
)
fixed_comparison = fixed_comparison.sort_values("model_order")

display(
    fixed_comparison[
        [
            "display_model",
            "accuracy_mean",
            "f1_mean",
            "roc_auc_mean",
            "precision_mean",
            "recall_mean",
        ]
    ].round(3)
)

In [ ]:
plot_frame = fixed_comparison[
    fixed_comparison["display_model"] != "dummy_most_frequent"
].copy()

plt.figure(figsize=(10, 7))
plt.barh(plot_frame["display_model"], plot_frame["f1_mean"])
plt.xlabel("Test F1, fixed threshold 0.5")
plt.ylabel("Model")
plt.title("Fixed-threshold test F1 by model")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## Threshold-tuned observations

These rows are worth keeping in the report, but the final model is still the fixed-threshold fused model.

In [ ]:
tuned_models = [
    "soft_label_fused_tabular_local_cnn",
    "fused_tabular_local_transit_set_cnn",
    "rescue_stacked_logistic_regression",
    "two_stage_rescue_gate",
]

tuned_comparison = comparison[
    (comparison["metric_variant"] == TUNED_VARIANT)
    & (comparison["split"] == "test")
    & (comparison["display_model"].isin(tuned_models))
].copy()

tuned_comparison["model_order"] = tuned_comparison["display_model"].map(
    {model: idx for idx, model in enumerate(tuned_models)}
)
tuned_comparison = tuned_comparison.sort_values("model_order")

display(
    tuned_comparison[
        [
            "display_model",
            "threshold_mean",
            "accuracy_mean",
            "f1_mean",
            "roc_auc_mean",
            "precision_mean",
            "recall_mean",
        ]
    ].round(3)
)

## Pairwise error analysis

Pairwise comparison answers the main question: did a new model fix more examples than it broke compared with the fused model?

In [ ]:
important_pairs = [
    "tabular_vs_fused",
    "tabular_local_features_vs_fused",
    "fused_vs_soft_label_fused_local",
    "fused_vs_candidate_weighted_fused_local",
    "fused_vs_three_class_fused_local",
    "fused_vs_two_stage_rescue_gate",
    "fused_vs_fused_transit_set",
    "fused_vs_fused_local_transit_set",
    "fused_vs_rescue_stacked",
    "fused_vs_selective_rescue_rule",
]

pairwise_fixed = pairwise[
    (pairwise["metric_variant"] == FIXED_VARIANT)
    & (pairwise["split"] == "test")
    & (pairwise["pair_id"].isin(important_pairs))
].copy()

pairwise_fixed["pair_order"] = pairwise_fixed["pair_id"].map(
    {pair: idx for idx, pair in enumerate(important_pairs)}
)
pairwise_fixed = pairwise_fixed.sort_values("pair_order")

display(
    pairwise_fixed[
        [
            "pair_id",
            "left_accuracy",
            "right_accuracy",
            "accuracy_delta_right_minus_left",
            "right_only_correct",
            "left_only_correct",
            "net_right_correct_gain",
            "changed_prediction_count",
        ]
    ].round(3)
)

In [ ]:
plt.figure(figsize=(10, 6))
plt.barh(pairwise_fixed["pair_id"], pairwise_fixed["net_right_correct_gain"])
plt.axvline(0, linestyle="--", linewidth=1)
plt.xlabel("Net right-correct gain")
plt.ylabel("Pair")
plt.title("Fixed-threshold pairwise net gain")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## Final interpretation

Takeaway: the KOI table is strong, and local light-curve shape adds useful extra signal. The best result comes from fusing those two inputs directly.

Final model: `fused_tabular_local_cnn` at fixed threshold 0.5. The soft-label, candidate-weighted, three-class, rescue, and learned-gate runs helped explain the errors, but none clearly beat the fused model at the fixed threshold.